# NABirds -> ResNet-50 Fine-Tuning (Head Only)
This notebook builds a NABirds training dataset and fine-tunes only the final classification layer of ResNet-50.

What this does:
- Supports either the original 98-species target subset or the full 555-class NABirds-specific label set.
- Auto-normalizes name differences (e.g., `Grey`/`Gray`, hyphenation, plural forms) for the target-species mode.
- Merges NABirds subsets/variants into one species label only when target-species mode is active.
- Crops each image to NABirds bounding box.
- Resizes with aspect ratio preserved and pads to `240x240` using ImageNet mean color.
- Uses NABirds official train/test split and creates a validation split from train.


In [ ]:
import csv
from datetime import datetime
from pathlib import Path
import os
import re
import random
import time
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image, ImageOps

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from tqdm.auto import tqdm

try:
    import psutil
except ImportError:
    psutil = None


# ---------------------------------------------------------------------------
# Run-summary helper (shared by all stage save cells)
# ---------------------------------------------------------------------------
_SUMMARY_CSV = Path("artifacts/logs/run_summary.csv")
_SUMMARY_COLUMNS = [
    "run_group_id", "run_label", "run_started_at", "stage", "seed",
    "split_file", "batch_size", "num_epochs", "lr", "weight_decay",
    "label_smoothing", "best_val_acc", "test_loss", "test_acc",
    "test_time_s", "run_time_s", "checkpoint_path", "config_json", "timestamp",
]

def _append_run_summary(row: dict) -> None:
    """Append one row to artifacts/logs/run_summary.csv (creates file+header if missing)."""
    _SUMMARY_CSV.parent.mkdir(parents=True, exist_ok=True)
    write_header = not _SUMMARY_CSV.exists()
    with open(_SUMMARY_CSV, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=_SUMMARY_COLUMNS)
        if write_header:
            writer.writeheader()
        writer.writerow({k: row.get(k, "") for k in _SUMMARY_COLUMNS})
    print(f"Appended to {_SUMMARY_CSV}")


In [ ]:
# Paths and reproducibility
DATA_ROOT = Path("NABirds_Dataset/nabirds")
IMAGES_DIR = DATA_ROOT / "images"

SEED = 42
VAL_FRACTION = 0.10
BATCH_SIZE = 32
DEFAULT_NUM_WORKERS = 4

try:
    from IPython import get_ipython
    IN_NOTEBOOK = get_ipython() is not None
except Exception:
    IN_NOTEBOOK = False

# Notebook + spawn multiprocessing often fails to pickle custom Dataset classes.
NUM_WORKERS = 0 if IN_NOTEBOOK else DEFAULT_NUM_WORKERS
NUM_EPOCHS = 20 #8 was original
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 2e-4
TARGET_SIZE = 240
LABEL_SMOOTHING = 0.00

# Training mode switch:
# False -> original 98-species merged target list
# True  -> full 555-class NABirds-specific labels
USE_ALL_SPECIFIC = False

# Split and output names follow the selected training mode.
DEFAULT_TRAIN_TEST_SPLIT_FILE = "train_test_split.txt"
TARGET_SPECIES_SPLIT_FILE = "train_test_split_8020_target_species.txt"
ALL_SPECIFIC_SPLIT_FILE = "train_test_split_8020_all_specific.txt"
TRAIN_TEST_SPLIT_FILE = ALL_SPECIFIC_SPLIT_FILE if USE_ALL_SPECIFIC else TARGET_SPECIES_SPLIT_FILE
LABEL_NAMES_OUTPUT = "label_names_nabirds_all_specific.csv" if USE_ALL_SPECIFIC else "label_names.csv"
MODE_DESCRIPTION = "555 NABirds-specific classes" if USE_ALL_SPECIFIC else "98 merged target species"

# Defaults used for checkpoint naming tags
DEFAULT_WEIGHT_DECAY = 1e-4

# Unique run identifier — appended to every checkpoint saved this session
# so re-running the notebook never silently overwrites a previous checkpoint.
RUN_LABEL = "notebook"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_STARTED_AT = datetime.now().isoformat(timespec="seconds")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")
print(f"IN_NOTEBOOK={IN_NOTEBOOK}, NUM_WORKERS={NUM_WORKERS}")
print(f"Training mode: {MODE_DESCRIPTION}")
print(f"Split file: {TRAIN_TEST_SPLIT_FILE}")
print(f"Run ID: {RUN_ID}")


In [66]:
# Target species list (from your original notebook)
TARGET_SPECIES = [
    "American Robin",
    "Mourning Dove",
    "Blue Jay",
    "House Sparrow",
    "Canada Goose",
    "Northern Mockingbird",
    "American Crow",
    "Carolina Chickadee",
    "Red-tailed Hawk",
    "Northern Cardinal",
    "Killdeer",
    "Dark-eyed Junco",
    "White-breasted Nuthatch",
    "Bald Eagle",
    "Red-bellied Woodpecker",
    "Carolina Wren",
    "Song Sparrow",
    "Wild Turkey",
    "European Starling",
    "Tufted Titmouse",
    "Mallard",
    "Common Raven",
    "American Goldfinch",
    "House Wren",
    "Eastern Phoebe",
    "Common Yellowthroat",
    "Great Horned Owl",
    "Northern Flicker",
    "Grey Catbird",
    "White-throated Sparrow",
    "Chimney Swift",
    "Belted Kingfisher",
    "Red-winged Blackbird",
    "Laughing Gull",
    "House Finch",
    "Common Loon",
    "Great Crested Flycatcher",
    "Ruby-crowned Kinglet",
    "Hermit Thrush",
    "Wood Duck",
    "Yellow Warbler",
    "Chipping Sparrow",
    "Red-eyed Vireo",
    "Tree Swallow",
    "Cooper's Hawk",
    "Ovenbird",
    "Winter Wren",
    "Cedar Waxwing",
    "Snow Goose",
    "Brown Thrasher",
    "Eastern Screech Owl",
    "Downy Woodpecker",
    "Rock Pigeon",
    "Wood Thrush",
    "Red-breasted Nuthatch",
    "Common Grackle",
    "Eastern Wood-Pewee",
    "Pileated Woodpecker",
    "Brown-headed Cowbird",
    "American Woodcock",
    "Barred Owl",
    "Golden-crowned Kinglet",
    "Eastern Whip-poor-will",
    "Indigo Bunting",
    "Brown Creeper",
    "Fish Crow",
    "Barn Swallow",
    "Eastern Towhee",
    "Warbling Vireo",
    "Ruby-throated Hummingbird",
    "Field Sparrow",
    "Eastern Bluebird",
    "Hairy Woodpecker",
    "Baltimore Orioles",
    "Eastern Meadowlark",
    "Black-capped Chickadee",
    "Osprey",
    "Scarlet Tanager",
    "Eastern Kingbird",
    "Great Blue Heron",
    "Yellow-billed Cuckoo",
    "Red-headed Woodpecker",
    "Rose-breasted Grosbeak",
    "Yellow-bellied Sapsucker",
    "Black-and-white Warbler",
    "Willow Flycatcher",
    "Hooded Merganser",
    "American Redstart",
    "Green Heron",
    "Purple Martin",
    "Yellow-rumped Warbler",
    "American Kestrel",
    "Common Nighthawk",
    "Ruffed Grouse",
    "Common Merganser",
    "Great Egret",
    "Double-crested Cormorant",
    "Mute Swan",
    "Turkey Vulture",
    "Black Vulture",
]

if USE_ALL_SPECIFIC:
    print("USE_ALL_SPECIFIC=True: the TARGET_SPECIES list is ignored and all NABirds classes in the split are used.")
else:
    print(f"Requested target species: {len(TARGET_SPECIES)}")


Requested target species: 100


## Build class mapping
This cell builds the label mapping for the selected training mode. Target-species mode merges NABirds variants into one species label, while all-specific mode keeps each NABirds class as its own output label.


In [67]:
def canonicalize_name(name: str) -> str:
    # Remove variant suffixes like "(Adult male)", then normalize punctuation and spacing.
    name = re.sub(r"\s*\([^)]*\)\s*", " ", name)
    name = name.lower()
    name = name.replace("grey", "gray")
    name = name.replace("orioles", "oriole")
    name = name.replace("-", " ")
    name = name.replace("'", "")
    name = re.sub(r"[^a-z0-9 ]+", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name

# Load NABirds metadata
images = pd.read_csv(DATA_ROOT / "images.txt", sep=" ", names=["image_id", "image_rel_path"])
labels = pd.read_csv(DATA_ROOT / "image_class_labels.txt", sep=" ", names=["image_id", "class_id"])
splits = pd.read_csv(DATA_ROOT / TRAIN_TEST_SPLIT_FILE, sep=" ", names=["image_id", "is_train"])
bboxes = pd.read_csv(DATA_ROOT / "bounding_boxes.txt", sep=" ", names=["image_id", "x", "y", "w", "h"])
# Parse classes.txt robustly: first token is class_id, remainder is class_name.
class_rows = []
with open(DATA_ROOT / "classes.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        cid, cname = line.split(maxsplit=1)
        class_rows.append((int(cid), cname))
classes = pd.DataFrame(class_rows, columns=["class_id", "class_name"])

valid_class_ids = set(labels["class_id"].unique())
classes = classes[classes["class_id"].isin(valid_class_ids)].copy()

if USE_ALL_SPECIFIC:
    # Keep each NABirds class as its own output label.
    classes = classes.sort_values("class_id").reset_index(drop=True)
    final_species = classes["class_name"].astype(str).tolist()
    class_id_to_species_idx = {
        int(cid): idx for idx, cid in enumerate(classes["class_id"].tolist())
    }

    print(f"Using all specific NABirds classes: {len(final_species)}")
else:
    # Merge NABirds variants into the requested species list.
    classes["canon"] = classes["class_name"].map(canonicalize_name)

    species_to_class_ids = {}
    dropped_species = []
    normalization_report = {}

    for species in TARGET_SPECIES:
        canon = canonicalize_name(species)
        matched = classes.loc[classes["canon"] == canon, "class_id"].tolist()
        if matched:
            species_to_class_ids[species] = sorted(set(matched))
            # Show auto-normalized cases for transparency.
            canonical_forms = sorted(set(classes.loc[classes["canon"] == canon, "class_name"]))
            if species not in canonical_forms:
                normalization_report[species] = canonical_forms
        else:
            dropped_species.append(species)

    print(f"Requested species: {len(TARGET_SPECIES)}")
    print(f"Mapped species:    {len(species_to_class_ids)}")
    print(f"Dropped species:   {len(dropped_species)}")
    if dropped_species:
        print("Dropped (not in NABirds):", dropped_species)

    if normalization_report:
        print("\nAuto-normalized names:")
        for src, matched_names in normalization_report.items():
            print(f"- {src} -> {matched_names[0]}")

    final_species = sorted(species_to_class_ids.keys())
    species_to_idx = {name: i for i, name in enumerate(final_species)}

    class_id_to_species_idx = {}
    for species, class_ids in species_to_class_ids.items():
        y = species_to_idx[species]
        for cid in class_ids:
            class_id_to_species_idx[cid] = y

target_to_species = dict(enumerate(final_species))


Requested species: 100
Mapped species:    98
Dropped species:   2
Dropped (not in NABirds): ['Eastern Whip-poor-will', 'Willow Flycatcher']

Auto-normalized names:
- American Robin -> American Robin (Adult)
- House Sparrow -> House Sparrow (Female/Juvenile)
- Red-tailed Hawk -> Red-tailed Hawk (Dark morph)
- Northern Cardinal -> Northern Cardinal (Adult Male)
- Dark-eyed Junco -> Dark-eyed Junco (Oregon)
- Bald Eagle -> Bald Eagle (Adult, subadult)
- European Starling -> European Starling (Breeding Adult)
- Mallard -> Mallard (Breeding male)
- American Goldfinch -> American Goldfinch (Breeding Male)
- Common Yellowthroat -> Common Yellowthroat (Adult Male)
- Northern Flicker -> Northern Flicker (Red-shafted)
- Grey Catbird -> Gray Catbird
- White-throated Sparrow -> White-throated Sparrow (Tan-striped/immature)
- Red-winged Blackbird -> Red-winged Blackbird (Female/juvenile)
- Laughing Gull -> Laughing Gull (Breeding)
- House Finch -> House Finch (Adult Male)
- Common Loon -> Common Lo

In [68]:
# Final label summary for the selected mode
print(f"Final class count: {len(final_species)}")


Final class count: 98


## Assemble filtered sample table
This keeps only images belonging to the selected label set and attaches split + bounding box metadata.


In [69]:
df = images.merge(labels, on="image_id", how="inner")
df = df.merge(splits, on="image_id", how="inner")
df = df.merge(bboxes, on="image_id", how="inner")

df = df[df["class_id"].isin(class_id_to_species_idx)].copy()
df["target"] = df["class_id"].map(class_id_to_species_idx)
df["species"] = df["target"].map(target_to_species)
df["image_path"] = df["image_rel_path"].map(lambda p: str(IMAGES_DIR / p))

print("Filtered samples:", len(df))
print("Train/Test by official split:")
print(df["is_train"].value_counts().rename(index={1: "train", 0: "test"}))


Filtered samples: 14579
Train/Test by official split:
is_train
train    11663
test      2916
Name: count, dtype: int64


In [70]:
# Stratified validation split from official train split
# Normalize split dtype defensively (some pandas versions parse as object/string).
df = df.copy()
df["is_train"] = pd.to_numeric(df["is_train"], errors="coerce")
df = df[df["is_train"].isin([0, 1])].copy()
df["is_train"] = df["is_train"].astype(int)

train_df = df[df["is_train"] == 1].copy().reset_index(drop=True)
test_df = df[df["is_train"] == 0].copy().reset_index(drop=True)

if train_df.empty:
    raise RuntimeError("No training samples found after filtering. Check earlier mapping output and class matches.")

rng = np.random.default_rng(SEED)
val_indices = []

for _, group in train_df.groupby("target", sort=False):
    idxs = group.index.to_numpy(copy=True)
    rng.shuffle(idxs)

    if len(idxs) < 2:
        n_val = 0
    else:
        n_val = max(1, int(round(len(idxs) * VAL_FRACTION)))
        n_val = min(n_val, len(idxs) - 1)

    if n_val > 0:
        val_indices.extend(idxs[:n_val].tolist())

# Fallback: guarantee non-empty val split when possible.
if len(val_indices) == 0 and len(train_df) > 1:
    val_indices = [int(rng.integers(0, len(train_df)))]

val_mask = train_df.index.isin(val_indices)
val_df = train_df[val_mask].copy().reset_index(drop=True)
train_df = train_df[~val_mask].copy().reset_index(drop=True)

print(f"Train samples: {len(train_df)}")
print(f"Val samples:   {len(val_df)}")
print(f"Test samples:  {len(test_df)}")


Train samples: 10487
Val samples:   1176
Test samples:  2916


## Dataset and transforms
Each sample is cropped to the bird bounding box, resized with preserved aspect ratio, then padded to `240x240` using ImageNet mean color.


In [71]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
IMAGENET_PAD_RGB = tuple(int(round(c * 255)) for c in IMAGENET_MEAN)


def crop_resize_pad(img: Image.Image, bbox, size=240, pad_rgb=(124, 116, 104)) -> Image.Image:
    x, y, w, h = bbox
    x1 = max(0, int(np.floor(x)))
    y1 = max(0, int(np.floor(y)))
    x2 = min(img.width, int(np.ceil(x + w)))
    y2 = min(img.height, int(np.ceil(y + h)))

    if x2 <= x1 or y2 <= y1:
        cropped = img
    else:
        cropped = img.crop((x1, y1, x2, y2))

    scale = min(size / cropped.width, size / cropped.height)
    new_w = max(1, int(round(cropped.width * scale)))
    new_h = max(1, int(round(cropped.height * scale)))
    resized = cropped.resize((new_w, new_h), resample=Image.BILINEAR)

    pad_left = (size - new_w) // 2
    pad_top = (size - new_h) // 2
    pad_right = size - new_w - pad_left
    pad_bottom = size - new_h - pad_top

    return ImageOps.expand(resized, border=(pad_left, pad_top, pad_right, pad_bottom), fill=pad_rgb)


train_transform = transforms.Compose([
    # Random crop at 60-100% scale then resize back to TARGET_SIZE.
    # Forces the model to handle birds at varying scales/positions within the bbox crop.
    transforms.RandomResizedCrop(TARGET_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    # Colour jitter: modest but meaningful variation in lighting and colour balance.
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    # RandomErasing (Cutout): randomly masks 2-20% of the image area,
    # forcing the model to not rely on any single patch (e.g. distinctive head markings).
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.2)),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class NABirdsSpeciesDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform=None):
        self.df = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")
        img = crop_resize_pad(
            img,
            bbox=(row["x"], row["y"], row["w"], row["h"]),
            size=TARGET_SIZE,
            pad_rgb=IMAGENET_PAD_RGB,
        )

        if self.transform is not None:
            img = self.transform(img)

        target = int(row["target"])
        return img, target


In [72]:
train_ds = NABirdsSpeciesDataset(train_df, transform=train_transform)
val_ds = NABirdsSpeciesDataset(val_df, transform=eval_transform)
test_ds = NABirdsSpeciesDataset(test_df, transform=eval_transform)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print(f"Dataloaders ready: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")


Dataloaders ready: train=10487, val=1176, test=2916


## ResNet-50 setup (freeze backbone)
All layers are frozen except the final `fc` layer.


In [73]:
num_classes = len(final_species)

weights = models.ResNet50_Weights.IMAGENET1K_V2
model = models.resnet50(weights=weights)

for p in model.parameters():
    p.requires_grad = False

in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, num_classes),
)

model = model.to(DEVICE)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable_params:,} / {total_params:,}")


Trainable params: 200,802 / 23,708,834


In [74]:
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(model.fc.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


In [75]:
def fmt_gb(num_bytes: int) -> str:
    return f"{num_bytes / (1024 ** 3):.2f} GB"


def get_device_memory_label(device: torch.device) -> str:
    if device.type == "cuda":
        allocated = torch.cuda.memory_allocated(device)
        reserved = torch.cuda.memory_reserved(device)
        return f"alloc={fmt_gb(allocated)} reserved={fmt_gb(reserved)}"
    if device.type == "mps" and hasattr(torch, "mps"):
        # MPS memory telemetry is available on Apple Silicon builds.
        alloc = torch.mps.current_allocated_memory()
        driver = torch.mps.driver_allocated_memory()
        rec_max = torch.mps.recommended_max_memory()
        return f"alloc={fmt_gb(alloc)} driver={fmt_gb(driver)} max={fmt_gb(rec_max)}"
    return "n/a"


def run_epoch(model, loader, criterion, optimizer=None, split_name="train"):
    is_train = optimizer is not None
    model.train(is_train)

    proc = psutil.Process(os.getpid()) if psutil is not None else None

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    epoch_start = time.perf_counter()
    pbar = tqdm(loader, leave=False, desc=f"{split_name}")

    for images, targets in pbar:
        step_start = time.perf_counter()

        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, targets)
            if is_train:
                loss.backward()
                optimizer.step()

        if DEVICE.type == "cuda":
            torch.cuda.synchronize(DEVICE)
        elif DEVICE.type == "mps" and hasattr(torch, "mps"):
            torch.mps.synchronize()

        preds = logits.argmax(dim=1)
        running_loss += loss.item() * targets.size(0)
        running_correct += (preds == targets).sum().item()
        running_total += targets.size(0)

        step_time = time.perf_counter() - step_start
        avg_loss_so_far = running_loss / max(1, running_total)
        avg_acc_so_far = running_correct / max(1, running_total)

        rss_label = fmt_gb(proc.memory_info().rss) if proc is not None else "n/a"
        device_mem_label = get_device_memory_label(DEVICE)

        pbar.set_postfix(
            loss=f"{avg_loss_so_far:.4f}",
            acc=f"{avg_acc_so_far:.4f}",
            step_s=f"{step_time:.3f}",
            rss=rss_label,
            dev_mem=device_mem_label,
        )

    avg_loss = running_loss / max(1, running_total)
    avg_acc = running_correct / max(1, running_total)
    epoch_time_s = time.perf_counter() - epoch_start
    return avg_loss, avg_acc, epoch_time_s


best_val_acc = -1.0
best_state = None
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc, train_time = run_epoch(
        model, train_loader, criterion, optimizer=optimizer, split_name=f"train e{epoch:02d}"
    )
    val_loss, val_acc, val_time = run_epoch(
        model, val_loader, criterion, optimizer=None, split_name=f"val   e{epoch:02d}"
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "train_time_s": train_time,
        "val_time_s": val_time,
    })

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} train_time={train_time:.1f}s | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_time={val_time:.1f}s"
    )

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Loaded best checkpoint from training (best val_acc={best_val_acc:.4f})")

history_df = pd.DataFrame(history)
history_df



train e01:   0%|          | 0/328 [00:00<?, ?it/s]

val   e01:   0%|          | 0/37 [00:00<?, ?it/s]

Epoch 01 | train_loss=3.3528 train_acc=0.2882 train_time=108.5s | val_loss=2.5086 val_acc=0.4405 val_time=9.7s


train e02:   0%|          | 0/328 [00:00<?, ?it/s]

val   e02:   0%|          | 0/37 [00:00<?, ?it/s]

...

Epoch 19 | train_loss=0.8109 train_acc=0.7773 train_time=119.9s | val_loss=0.8434 val_acc=0.7670 val_time=10.7s


train e20:   0%|          | 0/328 [00:00<?, ?it/s]

val   e20:   0%|          | 0/37 [00:00<?, ?it/s]

Epoch 20 | train_loss=0.7833 train_acc=0.7781 train_time=119.7s | val_loss=0.8282 val_acc=0.7704 val_time=10.7s
Loaded best checkpoint from training (best val_acc=0.7704)


,epoch,train_loss,train_acc,val_loss,val_acc,train_time_s,val_time_s
0,1,3.352793,0.288166,2.508598,0.440476,108.453442,9.682058
1,2,2.204796,0.523696,1.935335,0.553571,109.042053,9.647565
2,3,1.758374,0.604558,1.616037,0.625000,110.321893,9.794272
3,4,1.513659,0.652808,1.408658,0.663265,109.668121,9.726359
4,5,1.376915,0.676933,1.306994,0.694728,109.280695,9.716971
5,6,1.258999,0.688948,1.216988,0.696429,109.366778,9.709341
6,7,1.173288,0.710785,1.159941,0.708333,109.072993,9.693668
7,8,1.110904,0.720320,1.107152,0.713435,115.105884,10.725635
8,9,1.051811,0.735959,1.044303,0.735544,119.573669,10.712940
9,10,1.016146,0.737675,1.002876,0.743197,119.692195,10.773598


In [76]:
test_loss, test_acc, test_time = run_epoch(
    model, test_loader, criterion, optimizer=None, split_name="test"
)
print(f"Test loss: {test_loss:.4f}")
print(f"Test acc:  {test_acc:.4f}")
print(f"Test time: {test_time:.1f}s")



test:   0%|          | 0/92 [00:00<?, ?it/s]

Test loss: 0.7726
Test acc:  0.7915
Test time: 26.5s


In [ ]:
# Save class names and model weights for Stage 1
OUTPUT_DIR = Path("artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def format_float_tag(x: float) -> str:
    # Use compact scientific notation, e.g. 2e-4.
    return f"{x:.0e}".replace("e-0", "e-").replace("e+0", "e+")


def split_ratio_tag(split_filename: str) -> str:
    name = Path(split_filename).stem.lower()
    m = re.search(r"(\d{2})(\d{2})", name)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        if a + b == 100:
            return f"{a}-{b}"
    return Path(split_filename).stem


ckpt_tags = []
if WEIGHT_DECAY != DEFAULT_WEIGHT_DECAY:
    ckpt_tags.append(f"decay_{format_float_tag(WEIGHT_DECAY)}")
if TRAIN_TEST_SPLIT_FILE != DEFAULT_TRAIN_TEST_SPLIT_FILE:
    ckpt_tags.append(f"tt_{split_ratio_tag(TRAIN_TEST_SPLIT_FILE)}")

CKPT_TAG_SUFFIX = ("_" + "_".join(ckpt_tags)) if ckpt_tags else ""

# RUN_LABEL + RUN_ID suffix ensures this run never overwrites a previous one.
_run_suffix = f"_{RUN_LABEL}_{RUN_ID}"

if USE_ALL_SPECIFIC:
    STAGE1_CKPT_NAME = f"resnet50_nabirds_head_only_all_specific{CKPT_TAG_SUFFIX}{_run_suffix}.pt"
    STAGE2_CKPT_NAME = f"resnet50_nabirds_layer4_finetuned_all_specific{CKPT_TAG_SUFFIX}{_run_suffix}.pt"
    STAGE3_CKPT_NAME = f"resnet50_nabirds_layer3_layer4_finetuned_all_specific{CKPT_TAG_SUFFIX}{_run_suffix}.pt"
else:
    STAGE1_CKPT_NAME = f"resnet50_nabirds_head_only{CKPT_TAG_SUFFIX}{_run_suffix}.pt"
    STAGE2_CKPT_NAME = f"resnet50_nabirds_layer4_finetuned{CKPT_TAG_SUFFIX}{_run_suffix}.pt"
    STAGE3_CKPT_NAME = f"resnet50_nabirds_layer3_layer4_finetuned{CKPT_TAG_SUFFIX}{_run_suffix}.pt"

label_names_path = OUTPUT_DIR / LABEL_NAMES_OUTPUT
stage1_ckpt_path = OUTPUT_DIR / STAGE1_CKPT_NAME
torch.save(model.state_dict(), stage1_ckpt_path)
pd.Series(final_species).to_csv(label_names_path, index=False, header=["species"])
print("Saved:", stage1_ckpt_path)
print("Saved:", label_names_path)
print("Mode:", MODE_DESCRIPTION)
print("Checkpoint tag suffix:", CKPT_TAG_SUFFIX or "(default)")

import json as _json
_append_run_summary({
    "run_group_id": RUN_ID,
    "run_label": RUN_LABEL,
    "run_started_at": RUN_STARTED_AT,
    "stage": "stage1",
    "seed": SEED,
    "split_file": TRAIN_TEST_SPLIT_FILE,
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "lr": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "label_smoothing": LABEL_SMOOTHING,
    "best_val_acc": best_val_acc,
    "test_loss": test_loss,
    "test_acc": test_acc,
    "test_time_s": test_time,
    "run_time_s": "",
    "checkpoint_path": str(stage1_ckpt_path),
    "config_json": _json.dumps({
        "mode": MODE_DESCRIPTION,
        "use_all_specific": USE_ALL_SPECIFIC,
        "num_epochs": NUM_EPOCHS,
        "lr": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "label_smoothing": LABEL_SMOOTHING,
        "batch_size": BATCH_SIZE,
        "seed": SEED,
        "split_file": TRAIN_TEST_SPLIT_FILE,
        "run_label": RUN_LABEL,
    }),
    "timestamp": datetime.now().isoformat(timespec="seconds"),
})


## Stage 2: Unfreeze `layer4` + `fc` and fine-tune
This stage starts from the saved head-only checkpoint in `artifacts/` and fine-tunes the second-to-last ResNet block (`layer4`) plus the classifier (`fc`).


In [78]:
# Stage-2 hyperparameters
STAGE2_EPOCHS = 10
STAGE2_LR = 2e-4
STAGE2_WEIGHT_DECAY = 2e-4

stage2_ckpt_path = OUTPUT_DIR / STAGE1_CKPT_NAME
if not stage2_ckpt_path.exists():
    raise FileNotFoundError(f"Missing checkpoint: {stage2_ckpt_path}. Run Stage 1 save cell first.")


In [79]:
# Rebuild model, load Stage-1 weights, then unfreeze layer4 + fc only.
stage2_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
stage2_model.fc = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(stage2_model.fc.in_features, len(final_species)),
)

state = torch.load(stage2_ckpt_path, map_location="cpu")
stage2_model.load_state_dict(state)

for p in stage2_model.parameters():
    p.requires_grad = False

for p in stage2_model.layer4.parameters():
    p.requires_grad = True
for p in stage2_model.fc.parameters():
    p.requires_grad = True

stage2_model = stage2_model.to(DEVICE)

stage2_trainable = [p for p in stage2_model.parameters() if p.requires_grad]
stage2_optimizer = torch.optim.AdamW(
    stage2_trainable,
    lr=STAGE2_LR,
    weight_decay=STAGE2_WEIGHT_DECAY,
)

trainable_params = sum(p.numel() for p in stage2_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in stage2_model.parameters())
print(f"Stage 2 trainable params: {trainable_params:,} / {total_params:,}")


Stage 2 trainable params: 15,165,538 / 23,708,834


In [80]:
# Stage-2 training loop
best_val_acc_stage2 = -1.0
best_state_stage2 = None
history_stage2 = []

for epoch in range(1, STAGE2_EPOCHS + 1):
    train_loss, train_acc, train_time = run_epoch(
        stage2_model,
        train_loader,
        criterion,
        optimizer=stage2_optimizer,
        split_name=f"s2 train e{epoch:02d}",
    )
    val_loss, val_acc, val_time = run_epoch(
        stage2_model,
        val_loader,
        criterion,
        optimizer=None,
        split_name=f"s2 val   e{epoch:02d}",
    )

    history_stage2.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "train_time_s": train_time,
        "val_time_s": val_time,
    })

    if val_acc > best_val_acc_stage2:
        best_val_acc_stage2 = val_acc
        best_state_stage2 = {k: v.cpu().clone() for k, v in stage2_model.state_dict().items()}

    print(
        f"Stage2 Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} train_time={train_time:.1f}s | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_time={val_time:.1f}s"
    )

if best_state_stage2 is not None:
    stage2_model.load_state_dict(best_state_stage2)
    print(f"Loaded best Stage-2 checkpoint (best val_acc={best_val_acc_stage2:.4f})")

history_stage2_df = pd.DataFrame(history_stage2)
history_stage2_df


s2 train e01:   0%|          | 0/328 [00:00<?, ?it/s]

s2 val   e01:   0%|          | 0/37 [00:00<?, ?it/s]

Stage2 Epoch 01 | train_loss=0.5090 train_acc=0.8476 train_time=142.8s | val_loss=0.3846 val_acc=0.8895 val_time=10.8s


s2 train e02:   0%|          | 0/328 [00:00<?, ?it/s]

s2 val   e02:   0%|          | 0/37 [00:00<?, ?it/s]

...

Stage2 Epoch 09 | train_loss=0.1122 train_acc=0.9637 train_time=126.8s | val_loss=0.2838 val_acc=0.9243 val_time=9.6s


s2 train e10:   0%|          | 0/328 [00:00<?, ?it/s]

s2 val   e10:   0%|          | 0/37 [00:00<?, ?it/s]

Stage2 Epoch 10 | train_loss=0.1018 train_acc=0.9682 train_time=126.9s | val_loss=0.2706 val_acc=0.9167 val_time=9.6s
Loaded best Stage-2 checkpoint (best val_acc=0.9243)


,epoch,train_loss,train_acc,val_loss,val_acc,train_time_s,val_time_s
0,1,0.509050,0.847621,0.384586,0.889456,142.771748,10.776522
1,2,0.324268,0.895394,0.354463,0.895408,142.699993,10.631412
2,3,0.252752,0.921427,0.303504,0.906463,142.588552,10.718341
3,4,0.195871,0.938305,0.308499,0.910714,142.736044,10.973179
4,5,0.176752,0.944693,0.272748,0.924320,130.319528,9.643172
5,6,0.149896,0.951559,0.279641,0.917517,127.013884,9.625755
6,7,0.122063,0.961858,0.301991,0.917517,126.982665,9.719895
7,8,0.112975,0.963193,0.278802,0.924320,127.089840,9.883494
8,9,0.112238,0.963669,0.283842,0.924320,126.805699,9.627842
9,10,0.101831,0.968246,0.270625,0.916667,126.883906,9.624663


In [ ]:
# Stage-2 test evaluation + save
test_loss_s2, test_acc_s2, test_time_s2 = run_epoch(
    stage2_model,
    test_loader,
    criterion,
    optimizer=None,
    split_name="s2 test",
)
print(f"Stage 2 test loss: {test_loss_s2:.4f}")
print(f"Stage 2 test acc:  {test_acc_s2:.4f}")
print(f"Stage 2 test time: {test_time_s2:.1f}s")

stage2_ckpt_out = OUTPUT_DIR / STAGE2_CKPT_NAME
torch.save(stage2_model.state_dict(), stage2_ckpt_out)
print("Saved:", stage2_ckpt_out)

import json as _json
_append_run_summary({
    "run_group_id": RUN_ID,
    "run_label": RUN_LABEL,
    "run_started_at": RUN_STARTED_AT,
    "stage": "stage2",
    "seed": SEED,
    "split_file": TRAIN_TEST_SPLIT_FILE,
    "batch_size": BATCH_SIZE,
    "num_epochs": STAGE2_EPOCHS,
    "lr": STAGE2_LR,
    "weight_decay": STAGE2_WEIGHT_DECAY,
    "label_smoothing": LABEL_SMOOTHING,
    "best_val_acc": best_val_acc_stage2,
    "test_loss": test_loss_s2,
    "test_acc": test_acc_s2,
    "test_time_s": test_time_s2,
    "run_time_s": "",
    "checkpoint_path": str(stage2_ckpt_out),
    "config_json": _json.dumps({
        "mode": MODE_DESCRIPTION,
        "use_all_specific": USE_ALL_SPECIFIC,
        "stage2_epochs": STAGE2_EPOCHS,
        "stage2_lr": STAGE2_LR,
        "stage2_weight_decay": STAGE2_WEIGHT_DECAY,
        "seed": SEED,
        "split_file": TRAIN_TEST_SPLIT_FILE,
        "run_label": RUN_LABEL,
    }),
    "timestamp": datetime.now().isoformat(timespec="seconds"),
})


## Stage 3: Unfreeze `layer3` + `layer4` + `fc` and fine-tune
This optional final stage starts from the saved Stage-2 checkpoint and unlocks `layer3` in addition to `layer4` and `fc`. A smaller learning rate is used because more of the backbone is trainable.


In [ ]:
# Stage-3 hyperparameters and model setup
STAGE3_EPOCHS = 8
STAGE3_LR = 1e-4
STAGE3_WEIGHT_DECAY = 2e-4

# STAGE2_CKPT_NAME and STAGE3_CKPT_NAME were already set in the Stage 1 save cell
# (both carry the same RUN_ID so stages within a session chain together correctly).
stage3_ckpt_path = OUTPUT_DIR / STAGE2_CKPT_NAME
if not stage3_ckpt_path.exists():
    raise FileNotFoundError(
        f"Missing Stage-2 checkpoint: {stage3_ckpt_path}. Run the Stage 2 save cell first."
    )

# Rebuild model, load Stage-2 weights, then unfreeze layer3 + layer4 + fc.
stage3_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
stage3_model.fc = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(stage3_model.fc.in_features, len(final_species)),
)

state = torch.load(stage3_ckpt_path, map_location="cpu")
stage3_model.load_state_dict(state)

for p in stage3_model.parameters():
    p.requires_grad = False

for p in stage3_model.layer3.parameters():
    p.requires_grad = True
for p in stage3_model.layer4.parameters():
    p.requires_grad = True
for p in stage3_model.fc.parameters():
    p.requires_grad = True

stage3_model = stage3_model.to(DEVICE)

stage3_trainable = [p for p in stage3_model.parameters() if p.requires_grad]
stage3_optimizer = torch.optim.AdamW(
    stage3_trainable,
    lr=STAGE3_LR,
    weight_decay=STAGE3_WEIGHT_DECAY,
)

trainable_params = sum(p.numel() for p in stage3_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in stage3_model.parameters())
print(f"Stage 3 trainable params: {trainable_params:,} / {total_params:,}")
print(f"Loading from: {stage3_ckpt_path}")
print(f"Saving to:    {OUTPUT_DIR / STAGE3_CKPT_NAME}")


In [83]:
# Stage-3 training loop
best_val_acc_stage3 = -1.0
best_state_stage3 = None
history_stage3 = []

for epoch in range(1, STAGE3_EPOCHS + 1):
    train_loss, train_acc, train_time = run_epoch(
        stage3_model,
        train_loader,
        criterion,
        optimizer=stage3_optimizer,
        split_name=f"s3 train e{epoch:02d}",
    )
    val_loss, val_acc, val_time = run_epoch(
        stage3_model,
        val_loader,
        criterion,
        optimizer=None,
        split_name=f"s3 val   e{epoch:02d}",
    )

    history_stage3.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "train_time_s": train_time,
        "val_time_s": val_time,
    })

    if val_acc > best_val_acc_stage3:
        best_val_acc_stage3 = val_acc
        best_state_stage3 = {k: v.cpu().clone() for k, v in stage3_model.state_dict().items()}

    print(
        f"Stage3 Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} train_time={train_time:.1f}s | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_time={val_time:.1f}s"
    )

if best_state_stage3 is not None:
    stage3_model.load_state_dict(best_state_stage3)
    print(f"Loaded best Stage-3 checkpoint (best val_acc={best_val_acc_stage3:.4f})")

history_stage3_df = pd.DataFrame(history_stage3)
history_stage3_df


s3 train e01:   0%|          | 0/328 [00:00<?, ?it/s]

s3 val   e01:   0%|          | 0/37 [00:00<?, ?it/s]

Stage3 Epoch 01 | train_loss=0.1385 train_acc=0.9553 train_time=161.5s | val_loss=0.2942 val_acc=0.9184 val_time=9.6s


s3 train e02:   0%|          | 0/328 [00:00<?, ?it/s]

s3 val   e02:   0%|          | 0/37 [00:00<?, ?it/s]

...

Stage3 Epoch 07 | train_loss=0.0660 train_acc=0.9794 train_time=161.7s | val_loss=0.2662 val_acc=0.9226 val_time=9.6s


s3 train e08:   0%|          | 0/328 [00:00<?, ?it/s]

s3 val   e08:   0%|          | 0/37 [00:00<?, ?it/s]

Stage3 Epoch 08 | train_loss=0.0672 train_acc=0.9801 train_time=161.3s | val_loss=0.2517 val_acc=0.9371 val_time=9.7s
Loaded best Stage-3 checkpoint (best val_acc=0.9371)


,epoch,train_loss,train_acc,val_loss,val_acc,train_time_s,val_time_s
0,1,0.138494,0.955278,0.294246,0.918367,161.520866,9.640230
1,2,0.112890,0.964241,0.252716,0.926020,160.977082,9.617511
2,3,0.095478,0.969581,0.271977,0.927721,161.125570,9.650835
3,4,0.088273,0.972442,0.286035,0.921769,160.960036,9.647780
4,5,0.086597,0.971584,0.287359,0.926020,161.226396,9.633707
5,6,0.080639,0.975207,0.256964,0.931973,161.300743,9.683612
6,7,0.065981,0.979403,0.266176,0.922619,161.729652,9.641144
7,8,0.067152,0.980071,0.251730,0.937075,161.346119,9.680814


In [ ]:
# Stage-3 test evaluation + save
test_loss_s3, test_acc_s3, test_time_s3 = run_epoch(
    stage3_model,
    test_loader,
    criterion,
    optimizer=None,
    split_name="s3 test",
)
print(f"Stage 3 test loss: {test_loss_s3:.4f}")
print(f"Stage 3 test acc:  {test_acc_s3:.4f}")
print(f"Stage 3 test time: {test_time_s3:.1f}s")

stage3_ckpt_out = OUTPUT_DIR / STAGE3_CKPT_NAME
torch.save(stage3_model.state_dict(), stage3_ckpt_out)
print("Saved:", stage3_ckpt_out)

import json as _json
_append_run_summary({
    "run_group_id": RUN_ID,
    "run_label": RUN_LABEL,
    "run_started_at": RUN_STARTED_AT,
    "stage": "stage3",
    "seed": SEED,
    "split_file": TRAIN_TEST_SPLIT_FILE,
    "batch_size": BATCH_SIZE,
    "num_epochs": STAGE3_EPOCHS,
    "lr": STAGE3_LR,
    "weight_decay": STAGE3_WEIGHT_DECAY,
    "label_smoothing": LABEL_SMOOTHING,
    "best_val_acc": best_val_acc_stage3,
    "test_loss": test_loss_s3,
    "test_acc": test_acc_s3,
    "test_time_s": test_time_s3,
    "run_time_s": "",
    "checkpoint_path": str(stage3_ckpt_out),
    "config_json": _json.dumps({
        "mode": MODE_DESCRIPTION,
        "use_all_specific": USE_ALL_SPECIFIC,
        "stage3_epochs": STAGE3_EPOCHS,
        "stage3_lr": STAGE3_LR,
        "stage3_weight_decay": STAGE3_WEIGHT_DECAY,
        "seed": SEED,
        "split_file": TRAIN_TEST_SPLIT_FILE,
        "run_label": RUN_LABEL,
    }),
    "timestamp": datetime.now().isoformat(timespec="seconds"),
})
